In [2]:
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from xgboost import XGBClassifier


In [3]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"

mobilevit_features = np.load(f"{MODEL_DIR}/mobilevit_features.npy")
swin_features = np.load(f"{MODEL_DIR}/swin_features.npy")
labels = np.load(f"{MODEL_DIR}/labels.npy")

print("MobileViT:", mobilevit_features.shape)
print("Swin:", swin_features.shape)
print("Labels:", labels.shape)


MobileViT: (6939, 640)
Swin: (6939, 768)
Labels: (6939,)


In [4]:
X = np.concatenate([mobilevit_features, swin_features], axis=1)
y = labels

print("Final feature shape:", X.shape)


Final feature shape: (6939, 1408)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


Train samples: 5551
Test samples: 1388


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=600,              # ↑ more trees
    max_depth=8,                   # ↑ learn deeper interactions
    learning_rate=0.03,            # ↓ smoother learning
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,            # controls overfitting
    gamma=0.1,                     # split regularization
    reg_alpha=0.1,                 # L1
    reg_lambda=1.0,                # L2
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=3, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=600,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [17]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

scale_pos_weight = class_weights[1] / class_weights[0]
print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 1.2150837988826817


In [18]:
xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=3, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=600,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = xgb_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.952449567723343
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       761
           1       0.95      0.95      0.95       627

    accuracy                           0.95      1388
   macro avg       0.95      0.95      0.95      1388
weighted avg       0.95      0.95      0.95      1388



In [28]:
joblib.dump(xgb_model, f"{MODEL_DIR}/xgb_mobilevit_swin.pkl")
print("XGBoost FINAL model saved ✅")


XGBoost FINAL model saved ✅


In [2]:
import os
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier


In [3]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

os.makedirs(RESULTS_DIR, exist_ok=True)

mobilevit_features = np.load(f"{MODEL_DIR}/mobilevit_features.npy")
swin_features = np.load(f"{MODEL_DIR}/swin_features.npy")
labels = np.load(f"{MODEL_DIR}/labels.npy")

X = np.concatenate([mobilevit_features, swin_features], axis=1)
y = labels

print("Final feature shape:", X.shape)


Final feature shape: (6939, 1408)


In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (4440, 1408)
Validation: (1111, 1408)
Test: (1388, 1408)


In [5]:
np.save(f"{RESULTS_DIR}/X_train_swinS.npy", X_train)
np.save(f"{RESULTS_DIR}/y_train_swinS.npy", y_train)

np.save(f"{RESULTS_DIR}/X_val_swinS.npy", X_val)
np.save(f"{RESULTS_DIR}/y_val_swinS.npy", y_val)

np.save(f"{RESULTS_DIR}/X_test_swinS.npy", X_test)
np.save(f"{RESULTS_DIR}/y_test_swinS.npy", y_test)

print("✅ Train/Val/Test splits saved")


✅ Train/Val/Test splits saved


In [6]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

scale_pos_weight = class_weights[1] / class_weights[0]
print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 1.215568862275449


In [7]:
xgb_model = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,   # ✅ NOW USED
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


In [8]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

print("✅ XGBoost training completed")


✅ XGBoost training completed


In [9]:
results = xgb_model.evals_result()

train_logloss = results["validation_0"]["logloss"]
val_logloss = results["validation_1"]["logloss"]

np.save(f"{RESULTS_DIR}/swinS_train_logloss.npy", train_logloss)
np.save(f"{RESULTS_DIR}/swinS_val_logloss.npy", val_logloss)

print("✅ Logloss curves saved")


✅ Logloss curves saved


In [10]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9438040345821326
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       761
           1       0.94      0.94      0.94       627

    accuracy                           0.94      1388
   macro avg       0.94      0.94      0.94      1388
weighted avg       0.94      0.94      0.94      1388

